In [ ]:
# Imports & camera helpers
try:
    get_ipython().run_line_magic('matplotlib', 'widget')  # Requires ipympl
except RuntimeError:
    # Fallback if widget backend unavailable
    get_ipython().run_line_magic('matplotlib', 'notebook')

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import RectangleSelector

# If ipympl (widget backend) missing, advise install
try:
    import ipympl  # noqa: F401
except ImportError:
    print("⚠ ipympl not installed – interactive ROI will still work with 'notebook' backend but without fancy widgets.\nInstall with: pip install ipympl")

from queue import Queue
from vimba_centroid_lab.camera_vimba import CameraController

# Single global frame queue
_q = Queue(maxsize=1)
cam = CameraController(_q)


def grab_single_frame(timeout=2.0):
    """Start camera, grab one frame, stop camera, return np.ndarray."""
    cam.start()
    try:
        frame = _q.get(timeout=timeout)
    finally:
        cam.stop()
    return frame

img = grab_single_frame()
print(f"Captured frame shape: {img.shape}, dtype: {img.dtype}")


RuntimeError: 'widget' is not a recognised GUI loop or backend name

In [ ]:
# Display full image & interactive ROI selector
ax_img = None  # will hold the AxesImage
roi_coords = None  # (x0, y0, x1, y1)

fig, ax = plt.subplots(figsize=(8, 6))
ax.set_title("Draw ROI (drag mouse)")
ax.axis('off')
ax_img = ax.imshow(img, cmap='gray')


def onselect(eclick, erelease):
    global roi_coords
    x0, y0 = int(eclick.xdata), int(eclick.ydata)
    x1, y1 = int(erelease.xdata), int(erelease.ydata)
    roi_coords = (min(x0, x1), min(y0, y1), max(x0, x1), max(y0, y1))
    print(f"ROI selected: {roi_coords}")


rs = RectangleSelector(
    ax,
    onselect,
    drawtype='box',
    useblit=True,
    button=[1],  # left mouse
    minspanx=5,
    minspany=5,
    spancoords='pixels',
    interactive=True,
)

plt.show()


In [ ]:
# Crop & view ROI (run after selecting)
if roi_coords is None:
    raise ValueError("Please draw an ROI in the cell above before running this cell.")

x0, y0, x1, y1 = roi_coords
roi = img[y0:y1, x0:x1].copy()
print(f"ROI shape: {roi.shape}")

plt.figure(figsize=(4, 4))
plt.title("ROI – interpolation='none' to see pixels")
plt.imshow(roi, cmap='gray', interpolation='none')
plt.axis('off')


In [ ]:
# Placeholder for analysis
# You can add any analysis you want here, e.g., centroid detection, intensity stats, etc.

mean_intensity = roi.mean()
print(f"Mean intensity in ROI: {mean_intensity:.2f}")
